In [1]:
import pandas as pd
import numpy as np
from zero_point import zpt
zpt.load_tables()

In [2]:
# Funciones vectorizadas para correccion de paralaje y movimiento propio
# Usando operaciones de numpy para máximo rendimiento

def get_rotation_vectorized(G):
    """
    Calcula wx, wy, wz de forma vectorizada para arrays de magnitudes.
    
    Parameters
    ----------
    G : np.ndarray
        Array de magnitudes G
        
    Returns
    -------
    tuple of np.ndarray
        (wx, wy, wz) arrays
    """
    wx = np.full_like(G, 0, dtype=np.float64)
    wy = np.full_like(G, 0, dtype=np.float64)
    wz = np.full_like(G, 0, dtype=np.float64)
    
    mask1 = G < 9
    wx[mask1], wy[mask1], wz[mask1] = -5, -3, 0
    
    mask2 = (G >= 9) & (G < 11)
    wx[mask2], wy[mask2], wz[mask2] = -10, -5, 2
    
    mask3 = (G >= 11) & (G < 13)
    wx[mask3], wy[mask3], wz[mask3] = -25, -15, 5
    
    return wx, wy, wz

def pm_correction_vectorized(mu_ra, mu_dec, ra, dec, wx, wy, wz):
    """
    Corrige movimiento propio de forma vectorizada usando operaciones numpy.
    
    Parameters
    ----------
    mu_ra, mu_dec : np.ndarray
        Movimientos propios en RA y Dec (mas/yr)
    ra, dec : np.ndarray
        Coordenadas ecuatoriales (grados)
    wx, wy, wz : np.ndarray
        Parámetros de rotación del sistema de referencia
        
    Returns
    -------
    tuple of np.ndarray
        (mu_ra_corr, mu_dec_corr) corregidos
    """
    # Conversión a radianes (vectorizado)
    ra_rad = np.deg2rad(ra)
    dec_rad = np.deg2rad(dec)
    
    # Pre-computar funciones trigonométricas
    sin_ra = np.sin(ra_rad)
    cos_ra = np.cos(ra_rad)
    sin_dec = np.sin(dec_rad)
    cos_dec = np.cos(dec_rad)
    
    # Correcciones (operaciones vectorizadas)
    dmu_ra = -wx * sin_ra + wy * cos_ra
    
    dmu_dec = (-wx * cos_ra * sin_dec 
               - wy * sin_ra * sin_dec 
               + wz * cos_dec)
    
    # Aplicar correcciones
    mu_ra_corr = mu_ra - dmu_ra
    mu_dec_corr = mu_dec - dmu_dec
    
    return mu_ra_corr, mu_dec_corr


In [23]:
df_gaia = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_shell\gaia_230.csv")
df_sp = pd.read_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\DatosTotales\all_fidelity.csv")

In [25]:
# aplicando correccion de paralaje
df_gaia["zp"] = zpt.get_zpt(
    df_gaia["phot_g_mean_mag"],
    df_gaia["nu_eff_used_in_astrometry"],
    df_gaia["pseudocolour"],
    df_gaia["ecl_lat"],
    df_gaia["astrometric_params_solved"]
)

c:\Users\nicob\anaconda3\Lib\site-packages\zero_point\zpt.py:215: UserWarning: The apparent magnitude of one or more of the sources is outside the expected range (6-21 mag). 
                Outside this range, there is no further interpolation, thus the values at 6 or 21 are returned.
  warnings.warn(
c:\Users\nicob\anaconda3\Lib\site-packages\zero_point\zpt.py:230: UserWarning: The nu_eff_used_in_astrometry of some of the 5p source(s) is outside the expected range (1.1-1.9 
                mag). Outside this range, the zero-point calculated can be seriously wrong.
  warnings.warn(
c:\Users\nicob\anaconda3\Lib\site-packages\zero_point\zpt.py:243: UserWarning: The pseudocolour of some of the 6p source(s) is outside the expected range (1.24-1.72 mag).
                 The maximum corrections are reached already at 1.24 and 1.72
  warnings.warn(


In [26]:
df_gaia["parallax_corrected"] = df_gaia["parallax"] - df_gaia["zp"]/1000.0

In [29]:
df_gaia.head()

,source_id,ra,dec,parallax,pmra,pmdec,ruwe,phot_g_mean_mag,bp_rp,radial_velocity,...,parallax_error,visibility_periods_used,phot_bp_mean_mag,phot_rp_mean_mag,nu_eff_used_in_astrometry,pseudocolour,ecl_lat,astrometric_params_solved,zp,parallax_corrected
0,138832313879044096,46.076795,35.864350,6.694639,6.033624,-27.336710,1.043566,15.661435,2.753020,-13.066293,...,0.048872,15,17.214370,14.461350,1.260937,NaN,17.778413,31,-0.054061,6.694693
1,138944429705118336,46.736132,36.147998,4.627168,45.396808,-19.805744,1.031152,14.599747,1.872160,26.450920,...,0.023367,14,15.523865,13.651705,1.360007,NaN,17.896349,31,-0.043037,4.627211
2,138969821551607552,46.725418,36.475977,10.614968,84.123814,-46.586764,1.233279,13.714395,1.990878,53.810375,...,0.024990,13,14.714951,12.724072,1.345367,NaN,18.213029,31,-0.043869,10.615012
3,139005624398868864,46.089722,36.526846,4.764534,-8.688261,-21.951382,0.904957,18.244907,3.059847,NaN,...,0.163873,15,20.066843,17.006996,NaN,1.145332,18.409359,95,-0.047444,4.764581
4,139012835647884416,46.002892,36.718485,6.921768,14.016314,-33.248599,1.020420,17.206024,3.051111,NaN,...,0.093710,14,18.992582,15.941471,1.234228,NaN,18.612986,31,-0.049787,6.921818


In [30]:
df = df_gaia.merge(df_sp, on='source_id', how='left')

In [32]:
len(set(df_gaia.source_id).intersection(set(df_sp.source_id)))

88988

In [33]:
# Aplicar correcciones de rotacion y movimiento propio en una sola operacion vectorizada
# Esto es O(n) en lugar de O(n*m) con apply()

# Obtener arrays numpy para operaciones vectorizadas
G = df['phot_g_mean_mag'].values
ra = df['ra'].values
dec = df['dec'].values
mu_ra = df['pmra'].values
mu_dec = df['pmdec'].values

# Calcular parámetros de rotación
wx, wy, wz = get_rotation_vectorized(G)

# Aplicar correcciones de movimiento propio
mu_ra_corr, mu_dec_corr = pm_correction_vectorized(mu_ra, mu_dec, ra, dec, wx, wy, wz)

# Asignar resultados al dataframe
df['wx'] = wx
df['wy'] = wy
df['wz'] = wz
df['pmra_corrected'] = mu_ra_corr
df['pmdec_corrected'] = mu_dec_corr


In [34]:
# Validación y estadísticas de las correcciones aplicadas
print(f"Total de estrellas procesadas: {len(df):,}")
print(f"\nRangos de parámetros de rotación:")
print(f"  wx: [{df['wx'].min():.1f}, {df['wx'].max():.1f}] (media: {df['wx'].mean():.2f})")
print(f"  wy: [{df['wy'].min():.1f}, {df['wy'].max():.1f}] (media: {df['wy'].mean():.2f})")
print(f"  wz: [{df['wz'].min():.1f}, {df['wz'].max():.1f}] (media: {df['wz'].mean():.2f})")

print(f"\nCorrecciones de movimiento propio (mas/yr):")
print(f"  pmra_delta: media = {(df['pmra_corrected'] - df['pmra']).mean():.4f}, "
      f"std = {(df['pmra_corrected'] - df['pmra']).std():.4f}")
print(f"  pmdec_delta: media = {(df['pmdec_corrected'] - df['pmdec']).mean():.4f}, "
      f"std = {(df['pmdec_corrected'] - df['pmdec']).std():.4f}")

print(f"\nCorrelación entre componentes de movimiento propio original:")
print(f"  r(pmra, pmdec) = {np.corrcoef(df['pmra'], df['pmdec'])[0,1]:.4f}")
print(f"\nCorrelación entre componentes de movimiento propio corregidas:")
print(f"  r(pmra_corr, pmdec_corr) = {np.corrcoef(df['pmra_corrected'], df['pmdec_corrected'])[0,1]:.4f}")


Total de estrellas procesadas: 8,167,931

Rangos de parámetros de rotación:
  wx: [-25.0, 0.0] (media: -1.13)
  wy: [-15.0, 0.0] (media: -0.66)
  wz: [0.0, 5.0] (media: 0.22)

Correcciones de movimiento propio (mas/yr):
  pmra_delta: media = 0.0296, std = 4.1328
  pmdec_delta: media = -0.2130, std = 2.5190

Correlación entre componentes de movimiento propio original:
  r(pmra, pmdec) = -0.0159

Correlación entre componentes de movimiento propio corregidas:
  r(pmra_corr, pmdec_corr) = -0.0166


In [35]:
# Benchmark: Comparación de eficiencia con la solución anterior
import time

# Crear dataframe de prueba con subset de datos para benchmark
sample_size = min(10000, len(df))
df_sample = df.iloc[:sample_size].copy()

# Método anterior (ineficiente con apply + lambda)
def benchmark_apply_method():
    def get_rotation(G):
        if G < 9:
            return -5, -3, 0
        elif G < 11:
            return -10, -5, 2
        elif G < 13:
            return -25, -15, 5
        else:
            return 0, 0, 0
    
    df_test = df_sample.copy()
    start = time.perf_counter()
    df_test[['wx', 'wy', 'wz']] = df_test['phot_g_mean_mag'].apply(lambda G: pd.Series(get_rotation(G)))
    t1 = time.perf_counter() - start
    return t1

# vectorizado
def benchmark_vectorized_method():
    start = time.perf_counter()
    G = df_sample['phot_g_mean_mag'].values
    wx, wy, wz = get_rotation_vectorized(G)
    t2 = time.perf_counter() - start
    return t2

t_apply = benchmark_apply_method()
t_vect = benchmark_vectorized_method()

print(f"Benchmark ({sample_size} filas):")
print(f"  Método con apply():     {t_apply*1000:.2f} ms")
print(f"  Método vectorizado:     {t_vect*1000:.2f} ms")
print(f"  Speedup:                {t_apply/t_vect:.1f}x más rápido")
print(f"\nEn dataset completo ({len(df):,} filas):")
print(f"  Estimado (vectorizado): {(t_vect * len(df) / sample_size)*1000:.0f} ms")


Benchmark (10000 filas):
  Método con apply():     3721.92 ms
  Método vectorizado:     1.01 ms
  Speedup:                3671.3x más rápido

En dataset completo (8,167,931 filas):
  Estimado (vectorizado): 828 ms


In [36]:
print(len(df),len(df[df['fidelity_v2']>0.5]))

8167931 40121


In [17]:
df[df['parallax']>10].drop(columns=['fidelity_v1','wx','wy','wz','parallax_corrected','pmra_corrected','pmdec_corrected']).to_csv(r'C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_shell\dataset_100pc.csv')

In [38]:
df.reset_index(drop=True).to_csv(r"C:\Users\nicob\One Drive Uniandes\OneDrive - Universidad de los Andes\Doctorado\proyecto\clusterization_project\data\datos_shell\gaia_parallax_250_fidelity.csv")

In [37]:
df.reset_index(drop=True)

,source_id,ra,dec,parallax,pmra,pmdec,ruwe,phot_g_mean_mag,bp_rp,radial_velocity,...,astrometric_params_solved,zp,parallax_corrected,fidelity_v2,fidelity_v1,wx,wy,wz,pmra_corrected,pmdec_corrected
0,138832313879044096,46.076795,35.864350,6.694639,6.033624,-27.336710,1.043566,15.661435,2.753020,-13.066293,...,31,-0.054061,6.694693,NaN,NaN,0.0,0.0,0.0,6.033624,-27.336710
1,138944429705118336,46.736132,36.147998,4.627168,45.396808,-19.805744,1.031152,14.599747,1.872160,26.450920,...,31,-0.043037,4.627211,NaN,NaN,0.0,0.0,0.0,45.396808,-19.805744
2,138969821551607552,46.725418,36.475977,10.614968,84.123814,-46.586764,1.233279,13.714395,1.990878,53.810375,...,31,-0.043869,10.615012,NaN,NaN,0.0,0.0,0.0,84.123814,-46.586764
3,139005624398868864,46.089722,36.526846,4.764534,-8.688261,-21.951382,0.904957,18.244907,3.059847,NaN,...,95,-0.047444,4.764581,NaN,NaN,0.0,0.0,0.0,-8.688261,-21.951382
4,139012835647884416,46.002892,36.718485,6.921768,14.016314,-33.248599,1.020420,17.206024,3.051111,NaN,...,31,-0.049787,6.921818,NaN,NaN,0.0,0.0,0.0,14.016314,-33.248599
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8167926,6778550143712867328,313.895072,-36.122071,11.713349,49.728996,-1.753296,1.004384,14.669379,2.390958,-4.145230,...,31,-0.049286,11.713398,NaN,NaN,0.0,0.0,0.0,49.728996,-1.753296
8167927,6778613739292053760,313.538468,-35.962284,5.022365,9.050341,-11.870507,3.939765,19.540274,0.991568,NaN,...,31,-0.013616,5.022379,NaN,NaN,0.0,0.0,0.0,9.050341,-11.870507
8167928,6778644285098195968,313.980695,-36.077103,5.904595,62.126071,-42.779672,1.002695,19.517890,3.103392,NaN,...,95,-0.001742,5.904596,NaN,NaN,0.0,0.0,0.0,62.126071,-42.779672
8167929,6778674079288447360,314.513379,-35.823604,4.661621,-25.567933,16.410468,0.854568,11.397521,0.847132,29.053432,...,31,-0.037775,4.661659,NaN,NaN,-25.0,-15.0,5.0,2.775373,16.354244
